In [1]:
# Install pygbif
!pip install pygbif

In [2]:
# Import the below libraries
import time
import zipfile
from getpass import getpass
from glob import glob

import pygbif.occurrences as occ
import pygbif.species as species
import requests

# Import the following libraries to work with reproducible file paths
import os
import pathlib

# Import the following libraries to work with tabular data
import pygbif.occurrences as occ
import pandas as pd
import pygbif.species as species

# Import the earthpy library to manage the data
import earthpy

INFO:NumExpr defaulting to 16 threads.


In [3]:
# Create the migration data directory in the home folder
miller_moth_dir = os.path.join(
    # The home directory is the earth analytics data directory
    pathlib.Path.home(),
    'earth-analytics',
    'data',
    '02-migration-Livian-Von-Dran',
    'migration'
)

os.makedirs(miller_moth_dir, exist_ok=True)

# Define the GBIF data directory
gbif_miller_dir = os.path.join(miller_moth_dir)

In [4]:
reset = False

# Securely request and store GBIF username, password, and email address
if (not ('GBIF_USER'  in os.environ)) or reset:
    os.environ['GBIF_USER'] = input('GBIF username:')

if (not ('GBIF_PWD'  in os.environ)) or reset:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
    
if (not ('GBIF_EMAIL'  in os.environ)) or reset:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

In [5]:
# Conduct a search query for the miller moth
species_info = species.name_lookup('Euxoa auxiliaris', rank='SPECIES')

# Obtain the first result
first_result = species_info['results'][0]

# Find the species key
species_key = first_result['nubKey']

# Check the search query output
first_result['species'], species_key

('Euxoa auxiliaris', 4301148)

In [ ]:
# Only download the data once
gbif_miller_pattern = os.path.join(gbif_miller_dir, "gbif_data")
if not glob(gbif_miller_pattern):
    # Only submit one request
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:
        # Submit query to GBIF
        gbif_query = occ.download([
            f'speciesKey = { species_key }',
            'hasCoordinate = True',
            f'year = 2023',
        ])
        # Take the first result
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]

    # Wait for the download to build
    dld_key = os.environ['GBIF_DOWNLOAD_KEY']
    wait = occ.download_meta(dld_key)['status']
    while not wait=='SUCCEEDED':
        wait = occ.download_meta(dld_key)['status']
        time.sleep(5)

    # Download the GBIF data
    dld_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'], 
        path=gbif_miller_dir)
    dld_path = dld_info['path']

    # Unzip the GBIF data
    with zipfile.ZipFile(dld_path) as dld_zip:
        dld_zip.extractall(path=gbif_miller_dir)
        
    # Clean up the GBIF .zip file
    os.remove(dld_path)

# Find the extracted .csv file path for the first result
gbif_miller_path = glob(gbif_miller_pattern) = 

INFO:Download file size: 30630 bytes
INFO:On disk at C:\Users\livth\earth-analytics\data\02-migration-Livian-Von-Dran\migration/0011026-251025141854904.zip


In [8]:
# Load the GBIF data
gbif_df = pd.read_csv(
    gbif_miller_path,
    delimiter='\t',
)
gbif_df.head

ValueError: Invalid file path or buffer object type: <class 'list'>